# Pascal VOC 2007+2012 - ResNet50 FP32 baseline on Kaggle

Notebook này train Faster R-CNN ResNet50-FPN FP32 baseline trên Pascal VOC theo protocol phổ biến:

- train = VOC2007 trainval + VOC2012 trainval
- val/test = VOC2007 test

Pipeline hiện tại của repo đọc COCO-format, nên notebook sẽ convert VOC XML -> COCO JSON ngay trong `/kaggle/working` rồi mới train.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/EchteAI')
BRANCH = 'exp/resnet50-hawq-compiler'
REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'

if REPO.exists():
    print('Repo exists, pulling latest code...')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', BRANCH], check=False)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'pyyaml'], cwd=str(REPO), check=True)

print('Repository:', REPO)


In [ ]:
from pathlib import Path

WORK_ROOT = Path('/kaggle/working/pascal_voc_resnet50_fp32')
VOC_COCO_ROOT = Path('/kaggle/working/pascal_voc_coco')
RUNTIME_CONFIG = Path('/kaggle/working/runtime_pascal_voc_resnet50_fp32.yaml')
EXPORT_DIR = Path('/kaggle/working/export_pascal_voc_resnet50_fp32')

WORK_ROOT.mkdir(parents=True, exist_ok=True)
VOC_COCO_ROOT.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def find_vocdevkit(root='/kaggle/input'):
    root = Path(root)
    candidates = []
    for path in root.rglob('VOCdevkit'):
        if (path / 'VOC2007').is_dir() and (path / 'VOC2012').is_dir():
            candidates.append(path)
    if not candidates:
        for path in root.rglob('*'):
            if path.is_dir() and (path / 'VOC2007').is_dir() and (path / 'VOC2012').is_dir():
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError('Không tìm thấy VOCdevkit/VOC2007/VOC2012 trong /kaggle/input')
    candidates = sorted({candidate.resolve() for candidate in candidates})
    return candidates[0]

VOC_ROOT = find_vocdevkit('/kaggle/input')
print('VOC_ROOT =', VOC_ROOT)
print('WORK_ROOT =', WORK_ROOT)


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-u', 'scripts/convert_pascal_voc_to_coco.py',
    '--voc-root', str(VOC_ROOT),
    '--output-dir', str(VOC_COCO_ROOT),
    '--train-sets', 'VOC2007:trainval', 'VOC2012:trainval',
    '--val-sets', 'VOC2007:test',
]

print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)

assert (VOC_COCO_ROOT / 'instances_train.json').exists()
assert (VOC_COCO_ROOT / 'instances_val.json').exists()
print('COCO annotations ready at:', VOC_COCO_ROOT)


In [ ]:
from pathlib import Path
import json
import yaml

base = yaml.safe_load((REPO / 'configs' / 'seadronessee_resnet50_fp32_2class_1080x1920.yaml').read_text())

base['dataset']['train_images'] = str(VOC_ROOT)
base['dataset']['train_annotations'] = str(VOC_COCO_ROOT / 'instances_train.json')
base['dataset']['val_images'] = str(VOC_ROOT)
base['dataset']['val_annotations'] = str(VOC_COCO_ROOT / 'instances_val.json')
base['dataset']['test_images'] = str(VOC_ROOT)
base['dataset']['test_annotations'] = str(VOC_COCO_ROOT / 'instances_val.json')
base['dataset']['ignore_category_ids'] = []
base['dataset']['num_classes'] = 21
base['dataset']['binary_collapse_foreground'] = False
base['dataset']['workers'] = 2

base['model']['backbone'] = 'resnet50'
base['model']['trainable_backbone_layers'] = 5
base['model']['min_size'] = 800
base['model']['train_min_sizes'] = [640, 672, 704, 736, 768, 800]
base['model']['max_size'] = 1333
base['model']['anchor_statistics_min_size'] = 800

base['training']['batch_size'] = 1
base['training']['fp32_batch_size'] = 1
base['training']['fp32_epochs'] = 24
base['training']['warmup_iterations'] = 200
base['training']['print_frequency'] = 50
base['training']['epoch_benchmark_images'] = 100

base['output']['directory'] = str(WORK_ROOT)
base['output']['fp32_best'] = str(WORK_ROOT / 'fp32_best.pt')
base['output']['fp32_last'] = str(WORK_ROOT / 'fp32_last.pt')
base['output']['qat_best'] = str(WORK_ROOT / 'qat_best.pt')
base['output']['qat_last'] = str(WORK_ROOT / 'qat_last.pt')
base['output']['int8_model'] = str(WORK_ROOT / 'selective_int8.pt')
base['output']['evaluation_json'] = str(WORK_ROOT / 'evaluation.json')
base['output']['benchmark_json'] = str(WORK_ROOT / 'benchmark.json')
base['output']['epoch_benchmarks'] = str(WORK_ROOT / 'epoch_benchmarks.json')

RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False, allow_unicode=True), encoding='utf-8')

train_ann = json.loads((VOC_COCO_ROOT / 'instances_train.json').read_text())
val_ann = json.loads((VOC_COCO_ROOT / 'instances_val.json').read_text())
print('Saved runtime config:', RUNTIME_CONFIG)
print('train images:', len(train_ann['images']))
print('train annotations:', len(train_ann['annotations']))
print('val images:', len(val_ann['images']))
print('val annotations:', len(val_ann['annotations']))
print('\n=== Runtime config preview ===')
print(RUNTIME_CONFIG.read_text())


In [ ]:
import subprocess
import sys

LIMIT = None          # ví dụ 500 để smoke test
EPOCHS_THIS_RUN = 1   # tăng lên nếu muốn chạy nhiều epoch trong một session
MAX_STEPS = None      # ví dụ 1000 để cắt ngắn mỗi epoch
SKIP_BENCHMARK = False
SKIP_VALIDATION = False
INIT_CHECKPOINT = ''  # điền path nếu muốn warm-start từ checkpoint khác
PARTIAL_INIT = False

cmd = [
    sys.executable, '-u', 'scripts/train_resnet50_fp32_baseline.py',
    '--config', str(RUNTIME_CONFIG),
    '--epochs-this-run', str(EPOCHS_THIS_RUN),
]
if LIMIT is not None:
    cmd += ['--limit', str(LIMIT)]
if MAX_STEPS is not None:
    cmd += ['--max-steps', str(MAX_STEPS)]
if SKIP_BENCHMARK:
    cmd += ['--skip-benchmark']
if SKIP_VALIDATION:
    cmd += ['--skip-validation']
if INIT_CHECKPOINT:
    cmd += ['--init-checkpoint', INIT_CHECKPOINT]
    if PARTIAL_INIT:
        cmd += ['--partial-init-checkpoint']

print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO), check=True)


In [ ]:
from pathlib import Path
import json
import shutil

for name in [
    'fp32_best.pt',
    'fp32_last.pt',
    'evaluation.json',
    'benchmark.json',
    'epoch_benchmarks.json',
]:
    src = WORK_ROOT / name
    if src.exists():
        dst = EXPORT_DIR / name
        shutil.copy2(src, dst)
        print('copied:', dst)

print('\nWORK_ROOT =', WORK_ROOT)
if (WORK_ROOT / 'epoch_benchmarks.json').exists():
    print('\n=== epoch_benchmarks.json ===')
    print((WORK_ROOT / 'epoch_benchmarks.json').read_text())
